# Fold check by continuation

Newton-Raphson can fail before the true fold, so the noses found in `storms` could be solver failures. This notebook
follows the solution path of every storm with pseudo-arclength continuation, which walks through the fold instead of
stopping at it, and examines whether the smallest singular value of the Jacobian goes to zero there.

Runtime: CPU, about 0.3 GB of downloads, 10 to 20 minutes. Output: `fold_check.zip`.

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "fold_check"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVEL_BASE, BASE_PARTS = 2, [0, 1]
N_BASE = 120
LAM_STEP, LAM_WIDTH, LAM_MIN, LAM_CAP = 0.02, 0.0025, 0.3, 2.0   # plain-Newton nose search, as in `storms`
START_BELOW = 0.05            # the continuation starts this far below the plain-Newton nose
H0, H_MIN, H_MAX, MAX_STEPS = 0.02, 2e-4, 0.05, 400   # arclength steps
SIGMA_OFFSETS = [0.0025, 0.005, 0.01, 0.02, 0.04]       # load factors below the fold where sigma_min is computed
TOL = 1e-8
MAX_DOWNLOAD_GB = 2
N_TEST_STORMS, STORM_TOTAL = 6, 5   # only used on a small test grid
K1_NB = 2000
SEED = 0
K1_STORMS_JSON = r"""[{"sid":910000001,"base":0,"removed_lines":[1540,1537,1259,1542,945,1296,1544,1466,1256,1543,1541,1084,3096,1367,1031,944,925,1369,1366],"lam_max_K1":1.645,"solvable_at_1":true},{"sid":910000002,"base":1,"removed_lines":[1590,1553,1591,1403,2088,1833,3170,1345,1831,1419,1346,1830,1567,1418,1282,1161,3216,2176,1570],"lam_max_K1":0.55,"solvable_at_1":false},{"sid":910000003,"base":2,"removed_lines":[2025,2031,2028,1948,2030,2029,1635,1753,2033,2102,1996,1946,2027,1755,2032,2026,1634,1752,2133],"lam_max_K1":1.4325,"solvable_at_1":true},{"sid":910000004,"base":4,"removed_lines":[525,396,499,755,552,762,333,498,395,394,1998,549,502,734,330,57,2154,578],"lam_max_K1":0.4325,"solvable_at_1":false},{"sid":910000005,"base":6,"removed_lines":[1632,1637,1633,1634,1635,2030,2028,1975,1941,1942,2032,2033,1974,1675,1623,2000,1756,1626],"lam_max_K1":1.26,"solvable_at_1":true},{"sid":910000006,"base":8,"removed_lines":[1143,1284,1473,1144,1068,1028,993,1067,992,1286,1285,991,1530,980,1468,1039,1038,1321,902,994],"lam_max_K1":1.69375,"solvable_at_1":true},{"sid":910000007,"base":10,"removed_lines":[2015,1690,2016,2529,2891,2890,1806,1805,1691,1689,2883,2461,2914,1854,2521,1809,1850,808],"lam_max_K1":1.6225,"solvable_at_1":true},{"sid":910000008,"base":11,"removed_lines":[2645,2644,2624,2648,2976,2647,2625,2975,2650,2605,2614,2651,2711,2628,3073,2767,2506,2352],"lam_max_K1":1.6575,"solvable_at_1":true},{"sid":910000009,"base":17,"removed_lines":[2480,2956,2228,2584,2226,2569,2227,2479,2466,2229,2583,2572,2328,2756,2655,2935,2473,2822],"lam_max_K1":1.61,"solvable_at_1":true},{"sid":910000010,"base":20,"removed_lines":[1130,1317,1316,1503,996,1089,1177,995,1090,1341,1582,1536,994,991,1094,993,1092,895],"lam_max_K1":1.7,"solvable_at_1":true},{"sid":910000011,"base":23,"removed_lines":[1140,1403,3170,3216,1550,1413,2088,1346,3217,1569,1900,1023,3218,3214,1324,1521,3161,1570],"lam_max_K1":0.56625,"solvable_at_1":false},{"sid":910000012,"base":24,"removed_lines":[640,639,807,691,784,642,644,643,782,792,684,2253,793,590,790,1691,563,589,692],"lam_max_K1":1.0225,"solvable_at_1":true},{"sid":910000013,"base":26,"removed_lines":[2891,2883,2374,2890,2529,2526,2893,2525,2290,2530,2889,2521,2365,2816,2527,2528,2291,2531],"lam_max_K1":1.7375,"solvable_at_1":true},{"sid":910000014,"base":27,"removed_lines":[1515,1514,1269,1268,1583,1398,1250,1052,1251,1441,1053,1042,1069,1039,1026,946,1051,1478],"lam_max_K1":1.635,"solvable_at_1":true},{"sid":910000015,"base":29,"removed_lines":[1632,1625,1975,1633,1635,1636,1637,1624,1626,2049,1847,2028,1993,1994,1873,2072,2038,1942],"lam_max_K1":1.2075,"solvable_at_1":true},{"sid":910000016,"base":32,"removed_lines":[253,199,175,165,136,275,283,282,128,955,3195,956,952,3194,957,3136,3048,134],"lam_max_K1":0.5025,"solvable_at_1":false},{"sid":910000017,"base":33,"removed_lines":[2357,2364,2365,2309,2883,2461,2249,2367,2462,2991,562,2891,2768,2890,2529,2464,2310,2374],"lam_max_K1":null,"solvable_at_1":false},{"sid":910000018,"base":41,"removed_lines":[2461,2364,2462,3004,2398,2357,2309,2883,2463,2911,2366,2306,2767,2689,2195,2529,2392,3005,2992],"lam_max_K1":0.7025,"solvable_at_1":false},{"sid":910000019,"base":47,"removed_lines":[3088,3041,3040,3179,3063,3020,3067,3018,3086,3066,3019,3012,3187,3142,3167,2587,2586,2588],"lam_max_K1":1.04,"solvable_at_1":true},{"sid":910000020,"base":51,"removed_lines":[2225,2222,2223,2472,2469,2471,2344,2468,2343,2473,2466,2470,2467,2740,2664,2560,2415,2350],"lam_max_K1":1.5525,"solvable_at_1":true},{"sid":910000021,"base":55,"removed_lines":[456,458,261,256,435,488,433,160,489,385,877,48,378,383,123,369,393,436,1349],"lam_max_K1":0.52375,"solvable_at_1":false},{"sid":910000022,"base":59,"removed_lines":[1836,1825,1834,1627,1628,2129,1990,1970,2022,2130,1972,1983,1692,2083,1712,1768,1906,1964],"lam_max_K1":1.465,"solvable_at_1":true},{"sid":910000023,"base":76,"removed_lines":[463,348,346,1956,365,2181,2178,2056,2179,311,2050,1665,2180,1663,1664,2055,2160,1658,2045,1750],"lam_max_K1":0.97125,"solvable_at_1":false},{"sid":910000024,"base":83,"removed_lines":[2018,2019,1785,3207,1779,1784,3190,1858,1700,1782,1783,1780,1699,1854,3191,1856,1702,1695],"lam_max_K1":1.4125,"solvable_at_1":true},{"sid":910000025,"base":84,"removed_lines":[494,495,496,592,594,593,595,597,744,596,585,658,590,591,682,728,589,651],"lam_max_K1":0.6325,"solvable_at_1":false},{"sid":910000026,"base":85,"removed_lines":[1261,1150,1148,1547,1280,1080,1203,1400,887,943,1168,1402,1115,1456,1204,1535,1169,851],"lam_max_K1":1.1875,"solvable_at_1":true},{"sid":910000027,"base":87,"removed_lines":[545,522,620,524,657,656,655,812,813,654,809,537,507,810,605,742,617,652],"lam_max_K1":0.6975,"solvable_at_1":false},{"sid":910000028,"base":92,"removed_lines":[128,133,262,165,134,1031,170,267,126,275,283,1029,204,1030,1033,1432,199,1467,956],"lam_max_K1":0.83,"solvable_at_1":false},{"sid":910000029,"base":95,"removed_lines":[2105,2110,1727,1725,2112,2066,1611,2111,2055,1699,1728,1698,1750,1763,1730,1789,1662,1700,1877],"lam_max_K1":1.3925,"solvable_at_1":true},{"sid":910000030,"base":98,"removed_lines":[1919,1627,2082,1836,2145,1835,2039,2067,1973,1730,2066,2068,2063,2055,2022,1611,2065,2064],"lam_max_K1":1.5575,"solvable_at_1":true},{"sid":910000031,"base":109,"removed_lines":[1760,2131,1918,2132,1755,1753,1754,296,1741,295,1947,2030,1997,1995,2028,1751,1672,1998],"lam_max_K1":0.6925,"solvable_at_1":false},{"sid":910000032,"base":116,"removed_lines":[1911,1553,1588,1833,1899,1403,1831,1832,1345,1901,2048,2176,1346,1680,2173,2182,1419,2046],"lam_max_K1":1.29625,"solvable_at_1":true},{"sid":910000033,"base":119,"removed_lines":[148,219,161,289,220,216,160,1228,856,1231,1220,1032,1234,1289,871,1593,171,256],"lam_max_K1":0.36,"solvable_at_1":false},{"sid":910000034,"base":120,"removed_lines":[849,851,1338,1148,1118,1261,1203,1280,1115,390,940,1360,1204,1350,939,1131,941,942],"lam_max_K1":1.475,"solvable_at_1":true},{"sid":910000035,"base":121,"removed_lines":[95,105,18,98,59,57,68,77,56,60,396,394,20,312,86,119,113,3],"lam_max_K1":0.4275,"solvable_at_1":false},{"sid":910000036,"base":123,"removed_lines":[1850,1803,1854,1799,1611,1806,1603,1804,2250,1858,1696,1969,1781,1856,1160,2249,1613,1779,2055],"lam_max_K1":1.3675,"solvable_at_1":true},{"sid":910000037,"base":126,"removed_lines":[1618,1855,1617,1613,1616,1854,1612,1621,1614,1850,1803,1805,1807,1611,1781,1780,2015,1690],"lam_max_K1":1.4175,"solvable_at_1":true},{"sid":910000038,"base":127,"removed_lines":[834,865,1228,1220,1234,1120,1233,1315,936,1247,937,1267,1248,1231,1119,289,856,935],"lam_max_K1":1.48375,"solvable_at_1":true},{"sid":910000039,"base":128,"removed_lines":[1210,1423,1209,1043,1204,1547,1116,1380,1180,1148,1514,1117,1515,1115,1382,1456,1269,1584],"lam_max_K1":0.8375,"solvable_at_1":false},{"sid":910000040,"base":132,"removed_lines":[353,1154,1353,1137,2055,1668,392,2113,1587,1136,2050,1662,1568,118,939,1741,941,1611],"lam_max_K1":1.4075,"solvable_at_1":true},{"sid":910000041,"base":134,"removed_lines":[42,13,45,44,412,63,40,101,38,334,39,52,37,379,87,380,403,60,405],"lam_max_K1":1.1,"solvable_at_1":true},{"sid":910000042,"base":136,"removed_lines":[3089,1258,3097,1257,945,1259,1548,3127,1256,946,3208,3130,3115,3113,2646,925,3100,947],"lam_max_K1":1.16,"solvable_at_1":true},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max_K1":0.92375,"solvable_at_1":false},{"sid":910000044,"base":145,"removed_lines":[1241,1380,1310,1381,1043,1400,1542,1397,1180,1547,1477,1059,1548,1515,1456,1537,1398,1213],"lam_max_K1":1.1725,"solvable_at_1":true},{"sid":910000045,"base":146,"removed_lines":[1669,1680,1621,1671,1963,1616,1777,2007,2013,1832,1831,2005,2069,2089,1645,1914,1960,1962],"lam_max_K1":0.75375,"solvable_at_1":false},{"sid":910000046,"base":147,"removed_lines":[3143,3060,3119,3058,3135,3136,3165,3167,3121,3065,3066,3018,3180,954,282,3185,956,955,3017],"lam_max_K1":0.46375,"solvable_at_1":false},{"sid":910000047,"base":149,"removed_lines":[826,1116,1118,1304,1148,1203,943,1202,1149,1577,1389,941,1395,1165,939,1261,940,1584,391],"lam_max_K1":1.5025,"solvable_at_1":true},{"sid":910000048,"base":151,"removed_lines":[1741,1750,1749,1662,1752,1725,2055,2026,1789,1353,1753,1754,2086,1660,1712,1768,2131,2141,2110],"lam_max_K1":0.95625,"solvable_at_1":false},{"sid":910000049,"base":154,"removed_lines":[1969,1928,1964,1762,2243,1765,1972,2250,2249,1934,1920,1970,1921,1930,1931,2077,1749,1973],"lam_max_K1":1.5175,"solvable_at_1":true},{"sid":910000050,"base":155,"removed_lines":[2578,2576,2436,2577,2574,2264,2575,2265,2739,2431,2268,2432,2658,2805,2435,2475,2433,2926,2266],"lam_max_K1":1.485,"solvable_at_1":true},{"sid":910000051,"base":164,"removed_lines":[1515,1399,1398,1514,1441,1268,1269,1477,1432,945,1381,1440,944,1250,946,1052,916,941],"lam_max_K1":1.51,"solvable_at_1":true},{"sid":910000052,"base":169,"removed_lines":[17,57,59,95,105,98,55,61,22,115,396,395,119,20,499,526,407,38],"lam_max_K1":1.0225,"solvable_at_1":true},{"sid":910000053,"base":173,"removed_lines":[1880,1876,1701,1879,2175,1702,1699,2068,1877,1881,1695,1698,1666,2182,1703,1704,2174,1668,1696],"lam_max_K1":1.2325,"solvable_at_1":true},{"sid":910000054,"base":174,"removed_lines":[177,178,141,216,171,247,144,220,1349,124,148,197,289,870,161,259,866,1554],"lam_max_K1":0.795,"solvable_at_1":false},{"sid":910000055,"base":175,"removed_lines":[482,322,305,324,306,387,342,419,414,454,385,339,378,384,383,344,360,116],"lam_max_K1":1.4675,"solvable_at_1":true},{"sid":910000056,"base":181,"removed_lines":[1977,2099,2090,819,2100,1929,2998,2992,1928,1920,2101,2103,2031,1715,1950,1980,2276,2249],"lam_max_K1":1.4575,"solvable_at_1":true},{"sid":910000057,"base":182,"removed_lines":[115,113,38,39,65,3,118,117,86,60,40,1,36,31,1137,61,334,940],"lam_max_K1":0.975,"solvable_at_1":false},{"sid":910000058,"base":187,"removed_lines":[1788,2113,1667,1666,1664,1878,1660,1879,1882,2174,2140,2180,1701,1876,1661,2179,2160,2181],"lam_max_K1":1.47,"solvable_at_1":true},{"sid":910000059,"base":188,"removed_lines":[1884,1885,1600,1758,1601,1673,1883,1599,1719,1753,1837,1988,1752,1986,1676,1757,1987,1720,1672],"lam_max_K1":1.45,"solvable_at_1":true},{"sid":910000060,"base":190,"removed_lines":[2196,2970,2957,2868,2858,2194,2921,2607,2195,2971,2972,2973,2230,2832,2400,2603,2988,2481],"lam_max_K1":1.5425,"solvable_at_1":true},{"sid":910000061,"base":191,"removed_lines":[772,579,754,764,527,762,396,1980,525,526,549,2154,551,395,1993,2116,502,61,1996],"lam_max_K1":0.69125,"solvable_at_1":false},{"sid":910000062,"base":194,"removed_lines":[2161,1283,1570,1324,1569,2160,1346,1703,1568,2173,1668,1667,2172,1665,2175,1227,2171,1663],"lam_max_K1":0.69625,"solvable_at_1":false},{"sid":910000063,"base":203,"removed_lines":[1488,1372,1594,1404,1145,1505,1138,1274,1422,1421,997,1285,841,1284,1146,1139,1333,1224],"lam_max_K1":1.49,"solvable_at_1":true},{"sid":910000064,"base":205,"removed_lines":[2816,2914,2890,2817,2374,2529,2883,2814,2636,2291,2365,2290,2895,2527,2673,2818,2521,2938],"lam_max_K1":1.495,"solvable_at_1":true},{"sid":910000065,"base":209,"removed_lines":[2850,2798,2561,2560,2565,2562,2509,2567,2764,2412,2469,2563,2899,2601,2468,2766,2566,2900,2411],"lam_max_K1":1.445,"solvable_at_1":true},{"sid":910000066,"base":214,"removed_lines":[734,773,774,775,820,601,731,670,818,602,573,588,568,587,676,671,571,598],"lam_max_K1":0.4875,"solvable_at_1":false},{"sid":910000067,"base":217,"removed_lines":[535,581,777,706,647,707,584,691,808,650,646,576,563,639,765,564,690,2852],"lam_max_K1":0.87875,"solvable_at_1":false},{"sid":910000068,"base":220,"removed_lines":[2881,2714,2713,2897,2788,2894,2896,2437,2819,2895,2464,2820,2668,2816,2906,2414,2636,2747],"lam_max_K1":1.0825,"solvable_at_1":true},{"sid":910000069,"base":223,"removed_lines":[2743,3035,3003,2440,3002,1787,2997,3079,2322,2354,2589,2588,2294,2772,2995,2587,3073,2353],"lam_max_K1":1.205,"solvable_at_1":true},{"sid":910000070,"base":227,"removed_lines":[966,963,964,1034,1156,1035,1155,1518,1236,1087,844,1147,1111,1516,1004,1081,1005,1002],"lam_max_K1":1.52,"solvable_at_1":true},{"sid":910000071,"base":228,"removed_lines":[175,209,165,169,168,275,282,183,133,3194,954,952,3195,262,956,134,955,3172,1508],"lam_max_K1":0.64875,"solvable_at_1":false},{"sid":910000072,"base":233,"removed_lines":[975,974,3209,3210,1326,824,1297,3124,3102,3116,3203,3186,1541,3114,3115,3201,3089,3026,3049],"lam_max_K1":1.01,"solvable_at_1":true},{"sid":910000073,"base":236,"removed_lines":[1713,1715,1711,1712,1716,1714,1709,1768,1920,2169,1717,1798,2102,1719,1720,2168,1999,2031,1797],"lam_max_K1":1.615,"solvable_at_1":true},{"sid":910000074,"base":237,"removed_lines":[1764,1928,1964,1929,1762,1765,1920,1970,1930,1971,1934,2243,1804,2250,1798,1803,1977,1766,1799],"lam_max_K1":1.5525,"solvable_at_1":true},{"sid":910000075,"base":240,"removed_lines":[2524,2938,2939,2521,2953,2523,2674,2678,2673,2954,2817,2689,2889,2339,2921,2408,2527,2525],"lam_max_K1":1.525,"solvable_at_1":true},{"sid":910000076,"base":243,"removed_lines":[839,841,1198,1594,1238,831,1488,1152,828,829,1047,1172,1237,1199,911,832,1422,1192],"lam_max_K1":1.5225,"solvable_at_1":true},{"sid":910000077,"base":245,"removed_lines":[2868,2867,2860,2727,2973,2922,2392,2726,2847,2607,2924,2918,2496,2971,2923,2386,2717,2925,2958],"lam_max_K1":1.50625,"solvable_at_1":true},{"sid":910000078,"base":247,"removed_lines":[1196,1197,1097,1098,1481,1379,1491,1114,1377,1374,1470,1361,904,1272,1406,1358,1337,1371],"lam_max_K1":1.575,"solvable_at_1":true},{"sid":910000079,"base":253,"removed_lines":[1789,1725,1763,1764,1722,1796,1726,1765,1768,1750,1712,1751,1798,1964,2130,1753,1713,2077],"lam_max_K1":1.5425,"solvable_at_1":true},{"sid":910000080,"base":254,"removed_lines":[2374,2883,2890,2016,2529,2889,2461,2914,2530,2291,2892,2527,2526,2015,2531,2522,2528,1690],"lam_max_K1":1.55,"solvable_at_1":true},{"sid":910000081,"base":257,"removed_lines":[503,820,770,775,818,714,734,573,767,773,574,689,736,670,684,589,730,2099,1977],"lam_max_K1":0.39875,"solvable_at_1":false},{"sid":910000082,"base":263,"removed_lines":[550,549,552,755,2116,527,762,2115,2154,1998,564,808,2852,737,2253,2274,525,2165],"lam_max_K1":1.635,"solvable_at_1":true},{"sid":910000083,"base":264,"removed_lines":[940,942,389,939,938,390,941,118,1385,872,1571,117,1136,384,946,393,116,879],"lam_max_K1":1.4875,"solvable_at_1":true},{"sid":910000084,"base":267,"removed_lines":[1844,1845,1839,1737,2155,2034,1873,1868,2072,2035,1938,1871,1874,1838,1674,1736,2136,1739],"lam_max_K1":1.59,"solvable_at_1":true},{"sid":910000085,"base":269,"removed_lines":[2305,2300,2304,2303,2302,2789,2612,2278,2281,2610,2559,2279,2557,2399,2280,2400,2859,2352],"lam_max_K1":1.4175,"solvable_at_1":true},{"sid":910000086,"base":271,"removed_lines":[2678,2679,2690,2939,2930,2692,2630,2524,2693,2691,2494,2695,2522,2703,2673,2549,2986,2817,2698],"lam_max_K1":1.425,"solvable_at_1":true},{"sid":910000087,"base":280,"removed_lines":[1751,1749,1750,2050,1752,1725,1662,1353,1789,1754,2084,1755,2025,2086,1757,1722,1352,1765,2056],"lam_max_K1":1.5025,"solvable_at_1":true},{"sid":910000088,"base":282,"removed_lines":[769,770,767,570,683,779,820,775,591,590,818,780,690,771,730,589,692,691,808],"lam_max_K1":0.4775,"solvable_at_1":false},{"sid":910000089,"base":286,"removed_lines":[613,518,615,783,519,505,520,658,593,742,495,671,594,718,524,555,719,670],"lam_max_K1":1.19375,"solvable_at_1":true},{"sid":910000090,"base":287,"removed_lines":[1450,1318,1449,1448,1072,1292,1321,1322,1359,1405,1291,1341,1171,1237,1306,1357,1530,1363,1425],"lam_max_K1":1.6675,"solvable_at_1":true},{"sid":910000091,"base":296,"removed_lines":[2387,2872,2385,2717,2635,2729,2728,2407,2258,2726,2495,2724,2405,2409,2815,2868,2816,2481],"lam_max_K1":1.54625,"solvable_at_1":true},{"sid":910000092,"base":302,"removed_lines":[174,222,289,161,219,151,1232,215,856,216,1228,1235,1233,1349,835,1234,857,164],"lam_max_K1":0.3175,"solvable_at_1":false},{"sid":910000093,"base":306,"removed_lines":[2011,2009,2012,2013,2180,2174,2060,1843,1670,2181,1685,1777,1665,2173,1705,1990,1664,1866],"lam_max_K1":1.4975,"solvable_at_1":true},{"sid":910000094,"base":307,"removed_lines":[1671,1669,1777,1680,2075,2060,2007,1644,2073,1962,2010,2005,2009,2089,1960,1778,1704,2069],"lam_max_K1":1.58875,"solvable_at_1":true},{"sid":910000095,"base":308,"removed_lines":[2724,2839,2838,2409,2200,2535,2397,2725,2847,2846,2392,2410,2495,2918,2199,2551,2925,2774],"lam_max_K1":1.565,"solvable_at_1":true},{"sid":910000096,"base":316,"removed_lines":[1990,1866,1961,1865,1909,1960,1963,1962,2021,2023,1867,1644,1685,1653,1646,2014,2075,1682],"lam_max_K1":1.57,"solvable_at_1":true},{"sid":910000097,"base":317,"removed_lines":[3033,3031,3100,3099,3055,3045,3089,3122,3054,3056,3208,3209,3116,1258,2977,3115,3186,3176],"lam_max_K1":1.0625,"solvable_at_1":true},{"sid":910000098,"base":322,"removed_lines":[1494,1495,1593,1493,1492,1491,1444,1085,1032,1232,1445,926,1099,1081,1506,1082,289,956],"lam_max_K1":1.545,"solvable_at_1":true},{"sid":910000099,"base":329,"removed_lines":[220,215,216,1349,289,171,148,871,288,191,259,869,1231,158,393,877,856,1228,866],"lam_max_K1":0.56,"solvable_at_1":false},{"sid":910000100,"base":331,"removed_lines":[8,6,47,488,458,437,385,433,456,91,260,383,52,378,384,92,240,382,366],"lam_max_K1":1.07875,"solvable_at_1":true},{"sid":910000101,"base":332,"removed_lines":[2347,2694,2691,2703,2698,2345,2696,2692,2697,2699,2693,2701,2346,2702,2700,2808,2826,2343],"lam_max_K1":1.56,"solvable_at_1":true},{"sid":910000102,"base":333,"removed_lines":[2110,2105,1702,1722,2141,2112,2111,2055,1727,1700,1729,1603,1610,2145,2063,1699,2068,1695,1749],"lam_max_K1":1.36,"solvable_at_1":true},{"sid":910000103,"base":334,"removed_lines":[2879,2892,2746,2375,2747,2745,2788,2883,2891,2668,2016,2530,2365,2914,2522,1807,2436,2461,2291],"lam_max_K1":1.4975,"solvable_at_1":true},{"sid":910000104,"base":335,"removed_lines":[2977,2646,2975,2645,2978,2644,2587,2588,3127,3078,1548,2649,1456,3131,2278,3214,2586,1547,2650],"lam_max_K1":1.3925,"solvable_at_1":true},{"sid":910000105,"base":338,"removed_lines":[1708,1653,1704,1654,1657,1705,1656,1698,1699,1707,1650,1651,2008,1777,1840,1729,1889,1728],"lam_max_K1":1.34,"solvable_at_1":true},{"sid":910000106,"base":339,"removed_lines":[1820,2182,2068,1912,2175,1830,1878,2065,1730,2171,2064,2066,2063,1879,2173,1703,1831,1876],"lam_max_K1":1.5925,"solvable_at_1":true},{"sid":910000107,"base":341,"removed_lines":[2958,2196,2970,2973,2971,2868,2195,2607,2606,2188,2858,2921,2855,2626,2523,2231,2860,2988],"lam_max_K1":1.3925,"solvable_at_1":true},{"sid":910000108,"base":353,"removed_lines":[2898,2849,2216,2435,2320,2784,2319,2783,2436,2577,2931,2318,2315,2265,2788,2745,1807,2574],"lam_max_K1":1.1225,"solvable_at_1":true},{"sid":910000109,"base":354,"removed_lines":[898,897,899,1546,1070,1485,1428,890,1502,1053,1317,1545,1500,1431,892,1316,1176,1499],"lam_max_K1":1.6,"solvable_at_1":true},{"sid":910000110,"base":357,"removed_lines":[1513,1037,1512,1102,1151,1511,912,1213,1105,1336,1468,911,1212,915,997,1469,1100,1073],"lam_max_K1":1.635,"solvable_at_1":true},{"sid":910000111,"base":359,"removed_lines":[32,52,87,91,34,31,12,27,84,25,121,381,115,380,13,3,76,113,39],"lam_max_K1":0.81,"solvable_at_1":false},{"sid":910000112,"base":364,"removed_lines":[1400,1402,1180,1549,1456,1168,1458,1452,1169,3096,1309,1150,1457,3127,999,3130,2975,1210],"lam_max_K1":1.7,"solvable_at_1":true},{"sid":910000113,"base":365,"removed_lines":[957,3135,955,3141,952,282,958,956,283,960,1083,3165,3143,926,1440,3194,1507,1506],"lam_max_K1":1.47,"solvable_at_1":true},{"sid":910000114,"base":379,"removed_lines":[1018,1019,1167,1168,1519,1548,1587,1456,1352,1350,1584,1564,1458,1567,1353,1549,1400,1521,3127],"lam_max_K1":0.83625,"solvable_at_1":false},{"sid":910000115,"base":381,"removed_lines":[1102,1100,1105,1104,1103,1106,1513,1511,1469,1586,1508,1285,1314,1212,1185,1184,1313,912],"lam_max_K1":1.66,"solvable_at_1":true},{"sid":910000116,"base":383,"removed_lines":[1045,1210,1209,1043,1204,1564,1180,1116,1380,1547,1584,1395,1269,1149,1382,1117,1168,1548],"lam_max_K1":0.90375,"solvable_at_1":false},{"sid":910000117,"base":388,"removed_lines":[2771,2770,2769,2767,2316,2314,2768,2237,2996,2994,2626,2317,2995,2294,2293,2988,2934,2997],"lam_max_K1":1.645,"solvable_at_1":true},{"sid":910000118,"base":393,"removed_lines":[214,198,1302,1231,858,853,1289,855,1303,856,860,942,981,1207,989,879,1232,869],"lam_max_K1":1.4575,"solvable_at_1":true},{"sid":910000119,"base":396,"removed_lines":[1641,1643,1640,1642,1639,1870,1796,1619,1959,1874,1873,1768,1798,1712,2034,1839,1714,1598,1763],"lam_max_K1":1.4575,"solvable_at_1":true},{"sid":910000120,"base":397,"removed_lines":[1726,2105,1725,2111,1702,2141,1611,2066,1727,1647,1609,1749,1353,1699,2178,1648,1729,1695],"lam_max_K1":1.54625,"solvable_at_1":true}]"""

In [ ]:
# install (CPU only: no PyTorch, no graphkit)
import importlib.util, subprocess, sys
need = {"lightsim2grid": "lightsim2grid", "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""K1 core: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC truth along lambda up to the point where the AC equations stop having a solution (the
"nose"), a signature that the nose is genuine, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the step-1/6 core and power_flow_jacobian from the J1 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # the storm has no solution at lambda = 1: go down until one appears
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Drops toward 0 at a genuine nose."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""K2 core: pseudo-arclength continuation of the AC power flow in the load factor lambda, to locate the fold (nose)
of each storm independently of plain Newton's basin of convergence. Uses LambdaCase / sigma_min from the K1 core and
power_flow_jacobian / dsbus_dv from the earlier cores."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu


class ArcContinuation:
    """Unknowns z = [Va at PV+PQ buses, Vm at PQ buses, lambda]; F(z) = power mismatches at the scaled injections.
    Predictor along the unit tangent, corrector = Newton on F with the arclength constraint t . (z - z_pred) = 0."""

    def __init__(self, case):
        self.c = case
        self.pvpq = np.r_[case.pv, case.pq]
        self.n1, self.n2 = len(self.pvpq), len(case.pq)
        self.nx = self.n1 + self.n2
        self.Flam = np.r_[-case.P0[self.pvpq], case.Qd[case.pq]]          # dF/dlambda: F = S_calc - S(lambda)
        self.Vm_fixed = np.where(case.PQ, 1.0, case.d["Vm"])
        self.Va_fixed = np.radians(np.where(case.REF, case.d["Va"], 0.0))

    def pack(self, V):
        return np.r_[np.angle(V)[self.pvpq], np.abs(V)[self.c.pq]]

    def unpack(self, x):
        Va, Vm = self.Va_fixed.copy(), self.Vm_fixed.copy()
        Va[self.pvpq] = x[:self.n1]; Vm[self.c.pq] = x[self.n1:]
        return Vm * np.exp(1j * Va)

    def F(self, x, lam):
        V = self.unpack(x)
        S = V * np.conj(self.c.Y @ V) - self.c.S(lam)
        return np.r_[S.real[self.pvpq], S.imag[self.c.pq]]

    def J(self, x):
        return self.c.jacobian(self.unpack(x))

    def augmented(self, J, t):
        return sp.bmat([[J, sp.csc_matrix(self.Flam[:, None])],
                        [sp.csc_matrix(t[None, :self.nx]), sp.csc_matrix(np.array([[t[self.nx]]]))]], format="csc")

    def tangent(self, x, t_prev):
        A = self.augmented(self.J(x), t_prev)
        rhs = np.zeros(self.nx + 1); rhs[-1] = 1.0
        t = splu(A).solve(rhs)
        return t / np.linalg.norm(t)

    def corrector(self, z_pred, t, tol=1e-8, max_it=12):
        z = z_pred.copy()
        for it in range(max_it):
            F = self.F(z[:-1], z[-1])
            if np.max(np.abs(F)) < tol:
                return z, True, it
            A = self.augmented(self.J(z[:-1]), t)
            G = np.r_[F, t @ (z - z_pred)]
            try:
                z = z - splu(A).solve(G)
            except RuntimeError:
                return z, False, it
            if not np.all(np.isfinite(z)) or np.abs(z[self.n1:-1]).min() < 0.1:
                return z, False, it
        F = self.F(z[:-1], z[-1])
        return z, bool(np.max(np.abs(F)) < tol), max_it

    def trace_to_fold(self, V0, lam0, h0=0.02, h_min=2e-4, h_max=0.05, max_steps=400, tol=1e-8):
        """Follow the solution path from (V0, lam0) in the direction of rising lambda until the tangent's lambda
        component turns negative (the fold), then refine with smaller steps from the last pre-fold point.
        Returns dict: lam_fold (largest lambda reached, within O(h_min^2) of the fold), V_fold, path (lam, t_lam, h),
        status."""
        z = np.r_[self.pack(V0), lam0]
        t_prev = np.zeros(self.nx + 1); t_prev[-1] = 1.0
        t = self.tangent(z[:-1], t_prev)
        if t[-1] < 0:
            t = -t
        h, path, steps, refine = h0, [(lam0, float(t[-1]), 0.0)], 0, 0
        states = [(lam0, z.copy())]
        best = (lam0, z.copy())
        while steps < max_steps:
            steps += 1
            z_pred = z + h * t
            z_new, ok, its = self.corrector(z_pred, t, tol)
            if not ok:
                h *= 0.5
                if h < h_min:
                    return {"lam_fold": best[0], "V_fold": self.unpack(best[1][:-1]), "path": path, "states": states, "status": "corrector failed", "steps": steps}
                continue
            t_new = self.tangent(z_new[:-1], t)
            if t_new @ t < 0:
                t_new = -t_new
            if z_new[-1] > best[0]:
                best = (float(z_new[-1]), z_new.copy())
            if t_new[-1] < 0:          # passed the fold: go back and take a smaller step, until the step is tiny
                refine += 1
                h *= 0.25
                if h < h_min:
                    return {"lam_fold": best[0], "V_fold": self.unpack(best[1][:-1]), "path": path, "states": states, "status": "fold", "steps": steps}
                continue
            z, t = z_new, t_new
            path.append((float(z[-1]), float(t[-1]), h)); states.append((float(z[-1]), z.copy()))
            h = min(h_max, h * (1.5 if its <= 3 else 1.0) if its <= 5 else h * 0.5)
            if refine and h > 4 * h_min:
                h = 4 * h_min
        return {"lam_fold": best[0], "V_fold": self.unpack(best[1][:-1]), "path": path, "states": states, "status": "max steps", "steps": steps}


def state_below(arc, states, target, gap=0.002):
    """A path state at a load factor at most target - gap (the highest such), as a voltage vector to start Newton from."""
    cands = [(lam, z) for lam, z in states if lam <= target - gap]
    lam, z = max(cands, key=lambda p: p[0]) if cands else states[0]
    return arc.unpack(z[:-1])


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy, os
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def pick_storm(d, extra, rng):
    """(Only used on a small test grid.) `extra` more lines out around a random centre, grid kept in one piece."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    return (np.array(removed) if len(removed) == extra else None), center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)

## 1. Download: model folder and two N-2 data blocks

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
for p, s in all_files:
    m = RAW3_PAT.search(p) if p.startswith(f"data/test_{LEVEL_BASE}/") else None
    if m and int(m.group(2)) in BASE_PARTS:
        plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

## 2. Base points

In [ ]:
def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}


base_disk = scenario_ids(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"))
base_pool = np.array([s for s in base_disk if int(s) not in HARD2], dtype=np.int64)
BASE_IDS = np.sort(RNG.choice(base_pool, min(N_BASE, len(base_pool)), replace=False)).astype(np.int64)
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
tot = np.array([d["Pd"].sum() for d in BASE.values()])
print(f"{len(BASE)} N-{LEVEL_BASE} operating points as storm base points | {NB} buses | "
      f"total load {tot.min() / 1000:.1f}-{tot.max() / 1000:.1f} GW")

## 3. The storms, rebuilt

In [ ]:
# The 120 storms of notebook `storms` (base point and lines removed), rebuilt on the same data
K1_STORMS = json.loads(K1_STORMS_JSON)
CASES = []
if NB == K1_NB:
    need_ids = sorted({s["base"] for s in K1_STORMS})
    missing = [b for b in need_ids if b not in BASE]
    if missing:
        BASE.update(truth_dicts(LEVEL_BASE, missing))
    for s in K1_STORMS:
        d = BASE[s["base"]]
        rows = np.nonzero(np.isin(d["line"], np.array(s["removed_lines"])))[0]
        if len(rows) != len(s["removed_lines"]):
            raise RuntimeError(f"storm {s['sid']}: lines not found in base point {s['base']}")
        d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
        d2["on"][rows] = False
        d2["Va_dc"] = dc_angles(d2)
        case = LambdaCase(d2)
        V1, who = case.solve_any(1.0, [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)], 30, TOL)
        CASES.append({"sid": s["sid"], "base": s["base"], "d": d2, "case": case, "V1": V1, "solvable_at_1": V1 is not None,
                      "solvable_at_1_K1": s["solvable_at_1"], "lam_max_K1": s["lam_max_K1"]})
    agree = sum(c["solvable_at_1"] == c["solvable_at_1_K1"] for c in CASES)
    print(f"{len(CASES)} storms rebuilt | solvable at the actual load: {sum(c['solvable_at_1'] for c in CASES)} | agrees with K1 on {agree} of {len(CASES)}")
else:   # small test grid: build a few storms here
    rng = np.random.default_rng(SEED + 1)
    for j, b_id in enumerate(BASE_IDS[:N_TEST_STORMS]):
        d = BASE[int(b_id)]
        rem, centre = pick_storm(d, max(1, STORM_TOTAL - int((~d["on"]).sum())), rng)
        if rem is None:
            continue
        d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
        d2["on"][rem] = False
        d2["Va_dc"] = dc_angles(d2)
        case = LambdaCase(d2)
        V1, who = case.solve_any(1.0, [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)], 30, TOL)
        CASES.append({"sid": 920_000_000 + j, "base": int(b_id), "d": d2, "case": case, "V1": V1, "solvable_at_1": V1 is not None,
                      "solvable_at_1_K1": V1 is not None, "lam_max_K1": None})
    print(f"test grid: {len(CASES)} storms built")

## 4. Plain-Newton nose, then the fold by continuation

Unknowns: all free voltage angles and magnitudes plus the load factor. Predictor along the unit tangent of the solution
path, corrector = Newton with the arclength constraint. The fold is where the tangent's load-factor component changes
sign; the last point before it is refined with steps down to 2·10⁻⁴.

In [ ]:
t0 = time.perf_counter()
rows = []
for c in CASES:
    case = c["case"]
    r = {"sid": c["sid"], "base": c["base"], "solvable_at_1": c["solvable_at_1"], "lam_max_K1": c["lam_max_K1"]}
    res = find_lambda_max(case, c["V1"], step=LAM_STEP, width=LAM_WIDTH, lam_min=LAM_MIN, lam_cap=LAM_CAP, tol=TOL)
    r["lam_max_plain"] = res["lam_max"]; r["capped"] = res["capped"]
    if not np.isfinite(res["lam_max"]) or res["capped"]:
        r["status"] = "no nose below the cap" if res["capped"] else "no solution found at any load factor"
        rows.append(r); continue
    # start the continuation a little below the plain-Newton nose
    lam_start = max(res["lam_max"] - START_BELOW, LAM_MIN)
    T = truth_below_nose(case, res["lam_max"], res["V_max"], [lam_start], tol=TOL)
    V_start = T.get(lam_start, res["V_max"]); lam_start = lam_start if lam_start in T else res["lam_max"]
    arc = ArcContinuation(case)
    t1 = time.perf_counter()
    out = arc.trace_to_fold(V_start, lam_start, h0=H0, h_min=H_MIN, h_max=H_MAX, max_steps=MAX_STEPS, tol=TOL)
    r.update({"lam_fold": out["lam_fold"], "status": out["status"], "arc_steps": out["steps"], "arc_seconds": time.perf_counter() - t1,
              "fold_minus_plain": out["lam_fold"] - res["lam_max"]})
    # the Jacobian's smallest singular value on the way to the fold: sigma^2 should go linearly to 0 at the fold
    V_f = out["V_fold"]
    try:
        r["sigma_at_fold"] = sigma_min(case.jacobian(V_f))
        pts = []
        for dl in SIGMA_OFFSETS:
            lam = out["lam_fold"] - dl
            V2, ok, _, _ = case.solve(lam, state_below(arc, out["states"], lam), 30, TOL)
            if ok:
                pts.append((lam, sigma_min(case.jacobian(V2))))
        if len(pts) >= 3:
            lx, sy = np.array([p[0] for p in pts]), np.array([p[1] for p in pts]) ** 2
            a, b = np.polyfit(lx, sy, 1)                 # sigma^2 = a * lam + b -> 0 at lam = -b / a
            r["lam_sigma_zero"] = -b / a if a < 0 else np.nan
            r["sigma_fit_r2"] = 1 - np.sum((sy - (a * lx + b)) ** 2) / max(np.sum((sy - sy.mean()) ** 2), 1e-30)
            r["sigma_zero_minus_fold"] = r["lam_sigma_zero"] - out["lam_fold"]
            for (lam, sg) in pts:
                r[f"sigma_{out['lam_fold'] - lam:.4f}_below"] = sg
    except Exception as exc:
        r["sigma_error"] = repr(exc)[:100]
    rows.append(r)
NOSES = pd.DataFrame(rows)
NOSES.to_csv(Path(OUT_DIR) / "noses.csv", index=False)
print(f"done in {time.perf_counter() - t0:.0f} s")
ok = NOSES[NOSES.status == "fold"]
print(f"fold located by continuation for {len(ok)} of {len(NOSES)} storms (others: {NOSES[NOSES.status != 'fold'].status.value_counts().to_dict()})")
if len(ok):
    print(f"fold minus plain-Newton nose: median {ok.fold_minus_plain.median():+.4f}, 90th pct {ok.fold_minus_plain.quantile(0.9):+.4f}, max {ok.fold_minus_plain.max():+.4f} "
          f"(plain Newton located the nose to +/- {LAM_WIDTH})")
    if ok.lam_max_K1.notna().any():
        dk = ok.lam_fold - ok.lam_max_K1
        print(f"fold minus K1's nose: median {dk.median():+.4f}, 90th pct {dk.quantile(0.9):+.4f}, max {dk.max():+.4f}")
    if "sigma_zero_minus_fold" in ok:
        print(f"the Jacobian's smallest singular value, squared, extrapolates to zero at the fold {ok.sigma_zero_minus_fold.median():+.4f} "
              f"(median; IQR {ok.sigma_zero_minus_fold.quantile(0.25):+.4f} to {ok.sigma_zero_minus_fold.quantile(0.75):+.4f}); "
              f"fit R^2 median {ok.sigma_fit_r2.median():.3f}; sigma at the fold median {ok.sigma_at_fold.median():.2e} "
              f"against {ok[[c for c in ok.columns if c.endswith('_below')][-1]].median():.2e} at {SIGMA_OFFSETS[-1]} below it")
    display(NOSES.drop(columns=["base"]).describe().T[["count", "mean", "50%", "min", "max"]])

## 5. Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
if len(ok):
    axes[0].hist(ok.fold_minus_plain, bins=30, color="#1f5fa8")
    axes[0].set_xlabel("fold (continuation) minus nose (plain Newton), in load factor"); axes[0].set_ylabel("storms")
    axes[0].set_title("Where plain Newton stopped against the true fold", fontsize=10); axes[0].grid(alpha=0.3)
    below = sorted([c for c in ok.columns if c.endswith("_below")], key=lambda c: float(c.split("_")[1]))
    xs = np.array([float(c.split("_")[1]) for c in below])
    for _, r in ok.head(12).iterrows():
        ys = np.array([r[c] for c in below]) ** 2
        axes[1].plot(np.r_[0.0, xs], np.r_[r.sigma_at_fold ** 2, ys], "o-", ms=3, alpha=0.7)
    axes[1].set_xlabel("load factor below the fold"); axes[1].set_ylabel("smallest singular value of the Jacobian, squared")
    axes[1].set_title("Twelve storms: smallest singular value squared on the way to the fold", fontsize=10); axes[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig_fold_check.png", dpi=150); plt.show()

## 6. Summary and output files

In [ ]:
V = {}
if len(ok):
    V["FOLD_FOUND_FOR_MOST"] = bool(len(ok) >= 0.9 * (NOSES.status != "no nose below the cap").sum())
    V["PLAIN_NEWTON_NOSE_IS_THE_FOLD"] = bool(ok.fold_minus_plain.quantile(0.9) <= 0.01)
    if "sigma_zero_minus_fold" in ok:
        V["SIGMA_CONFIRMS_FOLD"] = bool(ok.sigma_zero_minus_fold.abs().median() <= 0.01 and ok.sigma_fit_r2.median() >= 0.9)
print("VERDICT")
for k, v in V.items():
    print(f"  {k}: {v}")
summary = {"config": {"LAM_STEP": LAM_STEP, "LAM_WIDTH": LAM_WIDTH, "START_BELOW": START_BELOW, "H0": H0, "H_MIN": H_MIN, "H_MAX": H_MAX,
                      "MAX_STEPS": MAX_STEPS, "SIGMA_OFFSETS": SIGMA_OFFSETS, "TOL": TOL, "SEED": SEED},
           "n_storms": len(NOSES), "n_fold": int(len(ok)), "verdict": V, "lightsim2grid": bool(HAVE_KLU), "platform": platform.platform()}
(Path(OUT_DIR) / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"\nSaved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB).")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass